---
title: "Gas–solid fluidized bed: online CFD–DEM coupling"
subtitle: "Blow gas up through a cylinder of grains until the bed unpacks and boils — the two solvers exchanging drag and void fraction every step — then run the real thing: a million 1 mm glass beads in air at 2.5 U~mf~, on the GPU, with a movie."
author: "Peclet"
date: "2026-07-06"
categories: [coupling, cfd-dem, dem, flow, fluidization, gidaspow, ergun, sdf, gpu, performance]
jupyter: python3
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-engineering/peclet-examples/blob/main/examples/fluidized-bed/index.ipynb){target="_blank"}
&nbsp;The two single-phase examples ([packed bed](../random-packed-bed/index.qmd), [drum](../rotating-drum/index.qmd)) are *offline* (dem → flow). This one is **online**: the CFD and DEM solvers run together and exchange forces every step, so it needs the `peclet-coupling` module (built from source — see *Reproduce*). The million-grain section needs a GPU build.

## What you'll learn

Everything so far coupled the two solvers **offline**: pack grains with `peclet.dem`, freeze them, then
solve the flow. A **fluidized bed** cannot be split that way — the grains move *because* of the gas and
the gas slows *because* of the grains. `peclet.coupling.CfdDem` runs the **unresolved point-particle
CFD–DEM** loop: every fluid step it deposits the grains' volume onto the grid (→ void fraction ε),
evaluates a drag law, pushes the reaction onto the fluid and the drag onto the grains, sub-steps the
DEM, and advances the fluid. With `porous=True` the fluid solves the **volume-averaged continuity**
`∂ε/∂t + ∇·(εu) = 0` — the gas *accelerates* through the packing (its velocity is the interstitial
one) and the pressure equation carries the drag on its diagonal, so a dense bed is unconditionally
stable. You will:

1. Build a **cylindrical vessel** whose signed distance field is used **twice** — a cut-cell IBM
   no-slip wall for the gas *and* a restitution+friction wall for the grains.
2. Drive it with a **gas inflow at the bottom** and an **outflow at the top**, close the momentum
   exchange with the **Gidaspow** drag law, and watch the bed **fluidize** once the gas clears
   minimum fluidization.
3. Run it at engineering conditions: **one million 1 mm glass beads in air at 2.5 U~mf~**, check the
   textbook criterion **ΔP ≈ bed weight per area**, and render the boiling bed into a movie —
   everything executes live in this notebook.

In [ ]:
#| label: bootstrap
#| code-summary: "Environment bootstrap (local build, or peclet + peclet-coupling)"
import importlib.util, os, subprocess, sys
_local = os.environ.get("PECLET_LOCAL_BUILD")
if _local:
    for p in _local.split(os.pathsep):
        sys.path.insert(0, p)
elif importlib.util.find_spec("peclet") is None:
    # peclet-coupling is sdist-only (it builds its Kokkos kernels from source against a Kokkos prefix);
    # on a plain runtime install the CPU family + the coupling extra.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peclet[cfd-dem]"], check=True)

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from peclet import flow, dem
from peclet.dem import build_wall_sdf
from peclet.coupling import CfdDem

plt.rcParams.update({"figure.dpi": 130, "font.size": 10, "axes.axisbelow": True,
                     "figure.facecolor": "white", "savefig.bbox": "tight"})
print("backends — flow:", flow.execution_space, " dem:", dem.execution_space)

## The setup — one cylinder, two solvers

The warm-up demo runs in **grid-cell units** (cell size `h = 1`; the million-grain section shows the
exact dictionary from SI). The vessel is a cylinder of radius `R` along `z`; the grains are
**spheres** of diameter `dp = 0.2` cells — the *unresolved* CFD–DEM requirement is a cell about
**five particle diameters** wide, so the void fraction each cell reports is a meaningful average over
many grains.

In [ ]:
NX = NY      = 8               # lateral grid
NZ           = 18              # tall column (bed + freeboard)
R, H_wall    = 2.5, 12.0       # vessel radius; grain-containment lid height (< NZ -> gas freeboard)
dp           = 0.2             # grain diameter -> cell / dp = 5
rp           = dp / 2
cx, cy       = NX / 2.0, NX / 2.0
rho_g, mu_g  = 1.0, 0.05       # gas
rho_p, grav  = 40.0, 2.0e-3    # grains (rho_p >> rho_g, like sand in air) + gravity
m_p          = rho_p * (4/3) * np.pi * rp**3

The **same radial SDF** — positive in the fluid, negative in the wall — is what both solvers collide
against. For the gas it becomes a cut-cell immersed no-slip wall; for the grains it becomes an SDF
wall with its own restitution and friction.

In [ ]:
def cylinder_flow_sdf():
    "Grid SDF (x-fastest): >0 in the fluid inside the vessel, <0 in the wall outside R."
    X, Y, _ = np.meshgrid(np.arange(NX) + .5, np.arange(NY) + .5, np.arange(NZ) + .5, indexing="ij")
    return (R - np.hypot(X - cx, Y - cy)).astype(np.float64)

def capped_cylinder_wall(pts):
    "Particle wall f(points)->distance, >0 in the void: inside R, above the distributor, below the lid."
    return np.minimum.reduce([R - np.hypot(pts[:, 0] - cx, pts[:, 1] - cy),
                              pts[:, 2] - 0.0, H_wall - pts[:, 2]])

**The gas** gets the cylinder as an immersed body plus an inflow floor and an outflow roof. Face codes
are `-x,+x,-y,+y,-z,+z`; the `-z` floor is an inflow at the superficial velocity `U`, the `+z` roof an
outflow, and the lateral faces no-slip (the fluid never reaches them — the cylinder confines it).

In [ ]:
def make_flow(U):
    s = flow.Solver(NX, NY, NZ)
    s.set_rho(rho_g); s.set_mu(mu_g); s.set_dt(0.05)
    s.set_domain_bc(4, 2, 0.0, 0.0, U)     # -z: inflow, gas up at U
    s.set_domain_bc(5, 3)                  # +z: outflow
    for f in (0, 1, 2, 3): s.set_domain_bc(f, 1)   # lateral no-slip
    s.set_pressure_pcg(True, 50, 1e-6)     # geometric-MG-preconditioned CG; 50 iters is plenty here
    s.set_solid(cylinder_flow_sdf().flatten(order="F"), True)   # cut-cell IBM no-slip vessel
    return s

::: {.callout-tip}
## Any length unit works — the halo follows the grain
You just set `radius = rp`. The DEM sizes its contact-search radius and periodic/rank **ghost band**
from the *actual* grain radius, so a grain 0.2 cells across — or `5e-4 m` in SI — gets a halo band
proportional to it, with no scaling gymnastics.
:::

**The grains** get gravity, a particle–particle material, and the capped-cylinder wall — a **bouncy
distributor** at the bottom (restitution 0.7, ≠ 1) and a **lid** that stops grains but is invisible to
the gas (which leaves through the outflow roof above it).

In [ ]:
def make_dem(pos):
    n = len(pos)
    d = dem.Simulation(int(2.2 * n) + 256)
    d.initialize(shape_type=1, radius=rp)        # sphere at the real grain radius (any unit works)
    d.set_domain((0, 0, 0), (NX, NY, NZ)); d.enable_periodicity(False, False, False)
    d.set_gravity(0, 0, -grav)
    d.set_material_params(0.8, 0.0, 0.2)          # particle-particle restitution / friction
    d.set_dt(0.05 / 20)
    build_wall_sdf(capped_cylinder_wall, ((0, 0, 0), (NX, NY, NZ)), resolution=64) \
        .add_to(d, restitution=0.7, friction=0.3)  # bouncy distributor + lid + side wall
    d.set_positions(np.c_[pos, np.full(n, 1.0 / m_p, np.float32)])
    d.set_velocities(np.zeros((n, 3), np.float32))
    return d

def initial_packing(n_bed=3.0, solid_frac=0.45):
    "Loose grains inside the vessel, z in (rp, n_bed]; they settle into a packed bed."
    vp = (4/3) * np.pi * rp**3
    npart = int(solid_frac * np.pi * R**2 * n_bed / vp)
    rng, out, n = np.random.default_rng(7), np.empty((npart, 3), np.float32), 0
    while n < npart:
        c = rng.uniform([cx - R, cy - R, rp], [cx + R, cy + R, n_bed], size=(npart, 3))
        c = c[(c[:, 0] - cx)**2 + (c[:, 1] - cy)**2 < (R - rp)**2]
        out[n:n + len(c)] = c[:npart - n]; n += len(c)
    return out

The coupling itself is one object. `porous=True` selects the volume-averaged fluid — the right model
for a bed; `drag="gidaspow"` the Ergun/Wen & Yu blend, converted to the Model-B form (`β/ε`)
internally because the gas here carries the full pressure gradient; gas convection (implicit
first-order upwind + a deferred-correction TVD term) is on by default. The void-fraction deposit is
trilinear and **wall-aware** — near the vessel wall a grain's volume is re-weighted onto the fluid
cells (partition of unity), so no hold-up leaks into the solid — and floored at the
random-close-packing voidage (`eps_min=0.4`), below which the drag correlations do not apply.

In [ ]:
pos0  = initial_packing()
print(f"{len(pos0)} grains, dp={dp} (cell/dp={1/dp:.0f}), R={R}")

def to_np(a):
    return a.get() if hasattr(a, "get") else np.asarray(a)

def bed_top(cpl):
    "95th-percentile grain height — the top of the bed."
    z = to_np(cpl._particles()[0])[:, 2]
    return float(np.percentile(z, 95)) if z.size else 0.0

## Fluidize it

Below minimum fluidization the gas trickles through a fixed bed; above it, the drag carries the grain
weight and the bed **unpacks and expands**. We drive it well above `U_mf` and record the bed height and
a couple of side-view snapshots.

In [ ]:
#| label: fig-fluidize
#| fig-cap: "The bed fluidizes: a settled packing (left) expands and boils as gas drives up through it (right); grains coloured by speed."
s   = make_flow(0.25)
d   = make_dem(pos0)
cpl = CfdDem(s, d, fluid_dt=0.05, mu=mu_g, rho=rho_g, radius=rp, drag="gidaspow",
             dem_substeps=20, periodic=(False, False, False), move_particles=True, porous=True)

h0, hist, snaps = bed_top(cpl), [], {}
for i in range(120):
    cpl.step()
    hist.append(bed_top(cpl))
    if i in (0, 119):
        p, v = cpl._particles()
        snaps[i] = (to_np(p).copy(), np.linalg.norm(to_np(v), axis=1))

fig, ax = plt.subplots(1, 3, figsize=(9, 3.4), gridspec_kw={"width_ratios": [1, 1, 1.3]})
for k, (i, title) in enumerate([(0, "settled bed"), (119, "fluidized")]):
    p, spd = snaps[i]
    sc = ax[k].scatter(p[:, 0], p[:, 2], c=spd, s=3, cmap="viridis", vmin=0, vmax=np.percentile(spd, 98))
    ax[k].add_patch(plt.Rectangle((cx - R, 0), 2 * R, H_wall, fill=False, ec="0.6", lw=1))
    ax[k].set(title=title, xlim=(0, NX), ylim=(0, NZ), xlabel="x", aspect="equal")
    ax[k].set_ylabel("z" if k == 0 else "")
ax[2].plot(np.arange(len(hist)) * 0.05, np.array(hist) / h0, lw=2)
ax[2].axhline(1, ls=":", c="0.6"); ax[2].set(xlabel="time", ylabel="bed height / initial", title="expansion")
plt.tight_layout()
print(f"bed height {h0:.2f} -> {hist[-1]:.2f}  (x{hist[-1]/h0:.2f})")

Sweeping the gas velocity traces the classic **fluidization curve**: flat while the bed is fixed, then
rising once `U` clears `U_mf`.

In [ ]:
#| label: fig-curve
#| fig-cap: "Fluidization curve: the bed height is flat while the gas only percolates, then climbs once the drag carries the grain weight (U > U_mf)."
def final_height(U, steps=80):
    s = make_flow(U); d = make_dem(pos0)
    c = CfdDem(s, d, fluid_dt=0.05, mu=mu_g, rho=rho_g, radius=rp, drag="gidaspow",
               dem_substeps=20, periodic=(False, False, False), move_particles=True, porous=True)
    for _ in range(steps): c.step()
    return bed_top(c)

Us = [0.0, 0.05, 0.10, 0.18, 0.30]
Hs = [final_height(U) for U in Us]
plt.figure(figsize=(4.6, 3.2))
plt.plot(Us, Hs, "o-", lw=2); plt.axhline(Hs[0], ls=":", c="0.6", label="fixed-bed height")
plt.xlabel("superficial gas velocity U"); plt.ylabel("bed height (95th pct)")
plt.title("fluidization curve"); plt.legend(); plt.tight_layout()
print("bed height vs U:", {U: round(h, 2) for U, h in zip(Us, Hs)})

## The real thing: a million 1 mm glass beads in air

Now at engineering conditions. **1 mm glass beads** (ρ~p~ = 2500 kg/m³) in **ambient air**
(ρ~g~ = 1.2 kg/m³, μ = 1.8·10⁻⁵ Pa·s), driven at **2.5 × U~mf~**. Minimum fluidization from the
Archimedes number (the Wen & Yu / Grace correlation):

In [ ]:
dp_m, rho_p_si, rho_g_si, mu_si, g_si = 1.0e-3, 2500.0, 1.2, 1.8e-5, 9.81
Ar    = rho_g_si * (rho_p_si - rho_g_si) * g_si * dp_m**3 / mu_si**2
Re_mf = np.sqrt(33.7**2 + 0.0408 * Ar) - 33.7
U_mf  = Re_mf * mu_si / (rho_g_si * dp_m)
U_si  = 2.5 * U_mf
print(f"Ar = {Ar:.3g}   Re_mf = {Re_mf:.1f}   U_mf = {U_mf:.2f} m/s   U = 2.5 U_mf = {U_si:.2f} m/s")

The solvers run in **cell units** — lengths in cells of `h = 5 d_p = 5 mm`, time in seconds,
densities in kg/m³. That one substitution (`x → x/h`, so `velocity/h`, `gravity/h`, `μ/h²`) preserves
every dimensionless group (Ar, Re, Froude), and the dictionary back to SI is just `length × h`,
`velocity × h`, `pressure × h²`. The vessel is **D = 10 cm** (100 grain diameters across), **40 cm**
tall; a million beads settle into a bed about **10 cm** deep (H/D ≈ 1).

In [ ]:
h_m = 5 * dp_m                                   # cell size, m
NXm = NYm = 22; NZm = 80                         # 11 x 11 x 40 cm box
Rm, cxm   = 10.0, 11.0                           # vessel R = 5 cm
H_lid     = 64.0                                 # grain lid (gas leaves above it)
Nbig      = 1_000_000
dpc, rpc  = dp_m / h_m, dp_m / h_m / 2           # grain: 0.2 cells
mu_c, g_c, U_c = mu_si / h_m**2, g_si / h_m, U_si / h_m
m_pc      = rho_p_si * (4/3) * np.pi * rpc**3
dt        = 1.0e-3                               # fluid step, s (20 DEM substeps of 50 us)
print(f"grid {NXm}x{NYm}x{NZm} = {NXm*NYm*NZm:,} cells   U = {U_c:.0f} cells/s   g = {g_c:.0f} cells/s^2")

In [ ]:
#| code-summary: "Build the SI vessel + pour and settle a million beads"
sB = flow.Solver(NXm, NYm, NZm)
sB.set_rho(rho_g_si); sB.set_mu(mu_c); sB.set_dt(dt)
sB.set_domain_bc(4, 2, 0.0, 0.0, U_c); sB.set_domain_bc(5, 3)
for f in (0, 1, 2, 3): sB.set_domain_bc(f, 1)
sB.set_pressure_pcg(True, 50, 1e-6)
Xm, Ym, _ = np.meshgrid(np.arange(NXm)+.5, np.arange(NYm)+.5, np.arange(NZm)+.5, indexing="ij")
sB.set_solid(np.asfortranarray((Rm - np.hypot(Xm-cxm, Ym-cxm)).astype(np.float64)).flatten(order="F"), True)

# pour: a jittered near-dense lattice inside the cylinder, then let the DEM settle it under gravity
sp = 1.05 * dpc
nxy = int(2*(Rm - 2*rpc)/sp)
xs = cxm - (nxy/2)*sp + (np.arange(nxy)+0.5)*sp
posB, k = [], 0
while len(posB) < Nbig:
    z = 2*rpc + (k + 0.5) * sp
    for ix in range(nxy):
        for iy in range(nxy):
            if (xs[ix]-cxm)**2 + (xs[iy]-cxm)**2 < (Rm-1.5*rpc)**2:
                posB.append((xs[ix], xs[iy], z))
    k += 1
posB = np.array(posB[:Nbig], np.float32)
posB[:, :2] += np.random.default_rng(7).uniform(-0.01, 0.01, (Nbig, 2)).astype(np.float32)

dB = dem.Simulation(int(1.15*Nbig) + 256)
dB.initialize(shape_type=1, radius=rpc)
dB.set_domain((0, 0, 0), (NXm, NYm, NZm)); dB.enable_periodicity(False, False, False)
dB.set_gravity(0, 0, -g_c)
dB.set_material_params(0.8, 0.0, 0.2)
dB.set_dt(dt / 20)
def wall_big(p):
    return np.minimum.reduce([Rm - np.hypot(p[:,0]-cxm, p[:,1]-cxm), p[:,2], H_lid - p[:,2]])
build_wall_sdf(wall_big, ((0,0,0), (NXm,NYm,NZm)), resolution=160).add_to(dB, restitution=0.7, friction=0.3)
dB.set_positions(np.c_[posB, np.full(Nbig, 1.0/m_pc, np.float32)])
dB.set_velocities(np.zeros((Nbig, 3), np.float32))

t0 = time.time()
for _ in range(2000):                            # 0.1 s of pure DEM settling
    dB.step(dt / 20)
pz = dB.get_positions()[:Nbig, 2]
print(f"settled in {time.time()-t0:.0f}s: bed top z95 = {np.percentile(pz,95)*h_m*100:.1f} cm")

Fluidize at 2.5 U~mf~ for **2.0 seconds** of physical time, recording a movie frame every 10 ms and
the two key diagnostics — the **bed pressure drop** and the **bed height** — as it runs. The textbook
check for a fluidized bed is `ΔP ≈ M·g/A` — the gas carries the weight of the solids. In a column this
narrow (H/D ≈ 1, D = 100 d~p~, wall friction 0.3) part of the weight is carried by the **walls**
(the Janssen effect, amplified by slugging), so the measured ΔP settles *below* the ideal line — a
well-documented feature of narrow laboratory columns, not a numerical artifact.

In [ ]:
cplB = CfdDem(sB, dB, fluid_dt=dt, mu=mu_c, rho=rho_g_si, radius=rpc, drag="gidaspow",
              dem_substeps=20, periodic=(False, False, False), move_particles=True, porous=True)

W_over_A = Nbig * m_pc * g_c / (np.pi * Rm**2) * h_m**2           # bed weight / area, Pa
inside = np.hypot(Xm[:, :, 0] - cxm, Ym[:, :, 0] - cxm) < Rm - 1  # cells inside the vessel

frames, dPs, beds = [], [], []
steps, every = 2000, 10                                           # 2.0 s, frame every 10 ms
t0 = time.time()
for i in range(steps):
    cplB.step()
    p = sB.get_p()
    dPs.append((np.nanmean(p[:, :, 1][inside]) - np.nanmean(p[:, :, -2][inside])) * h_m**2)
    if i % every == 0:
        pp, vv = cplB._particles()
        pp, vv = to_np(pp), to_np(vv)
        front = pp[:, 1] <= cxm                                   # front-half cutaway
        frames.append((pp[front].copy(), (np.linalg.norm(vv[front], axis=1) * h_m).astype(np.float32)))
        beds.append(np.percentile(pp[:, 2], 95) * h_m)
wall = time.time() - t0
print(f"{steps} coupled steps ({steps*dt:.1f} s physical) in {wall/60:.1f} min "
      f"({wall/steps*1e3:.0f} ms/step at {Nbig:,} grains)")
print(f"final dP = {dPs[-1]:.0f} Pa   bed weight/A = {W_over_A:.0f} Pa   ratio = {dPs[-1]/W_over_A:.2f}")

In [ ]:
#| label: fig-si-diag
#| fig-cap: "The engineering checks. Left: after the violent unpacking transient (the startup slug spikes ΔP to ~2.4 Mg/A), the bed pressure drop oscillates with the bubbling at ~60% of the bed-weight line — the balance is carried by wall friction in this narrow column. Right: the bed surface bubbles around its working height."
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(np.arange(len(dPs)) * dt, dPs, lw=1)
ax[0].axhline(W_over_A, ls="--", c="C3", label="bed weight / area")
ax[0].set(xlabel="time [s]", ylabel="ΔP over the bed [Pa]"); ax[0].legend()
ax[1].plot(np.arange(len(beds)) * every * dt, np.array(beds) * 100, lw=1.5, color="C1")
ax[1].set(xlabel="time [s]", ylabel="bed height z95 [cm]")
plt.tight_layout()

Render the recorded front-half cutaways with [PyVista](https://pyvista.org) — half a million visible
grains per frame, coloured by speed — and write the movie:

In [ ]:
#| code-summary: "PyVista offscreen renderer -> fluidized_bed_1M.mp4"
import pyvista as pv, imageio.v2 as imageio
pv.OFF_SCREEN = True
writer = imageio.get_writer("fluidized_bed_1M.mp4", fps=20, quality=8)
for pts, spd in frames:
    cloud = pv.PolyData(pts.astype(np.float32))
    cloud["speed [m/s]"] = spd
    pl = pv.Plotter(off_screen=True, window_size=(560, 912))
    pl.background_color = "white"
    pl.add_mesh(cloud, scalars="speed [m/s]", cmap="turbo", clim=(0.0, 1.5),
                point_size=2.4, render_points_as_spheres=True,
                scalar_bar_args=dict(vertical=True, position_x=0.86, position_y=0.30,
                                     height=0.45, width=0.07, title="speed [m/s]"))
    pl.add_mesh(pv.Cylinder(center=(cxm, cxm, 20), direction=(0, 0, 1),
                            radius=Rm, height=40), color="gray", opacity=0.05)
    pl.camera_position = [(cxm, -50, 21), (cxm, cxm, 14), (0, 0, 1)]
    pl.hide_axes()
    writer.append_data(pl.screenshot(return_img=True)); pl.close()
writer.close()
print(f"{len(frames)} frames -> fluidized_bed_1M.mp4")

<video controls autoplay loop muted playsinline width="100%" style="max-width:560px;display:block;margin:0 auto;border-radius:6px;">
  <source src="fluidized_bed_1M.mp4" type="video/mp4">
</video>

A **D/d~p~ = 100** column at 2.5 U~mf~ is a vigorously bubbling/slugging bed (1 mm glass sits near the
Geldart B/D boundary): gas voids nucleate at the distributor, coalesce as they rise, and erupt through
the surface, raining grains back down the walls. The startup is itself textbook: the packed bed
unpacks with one violent slug (the ΔP spike), overshoots, and collapses into sustained bubbling.

## Performance

The numbers above are the honest, *dense-bed*, fully-coupled cost on one RTX 5080: a **million-grain
coupled step in ~1.2 s** — 20 XPBD substeps on a packed, contact-rich bed (ArborX broad-phase +
narrow-phase + constraint solve, ~58 ms per substep ≈ 17 M grain-substeps/s at packing fraction ~0.6)
plus the volume-averaged fluid solve on the 39k-cell grid; a loosely packed bed steps in half that.
The whole 2-second movie run is ~40 minutes of wall clock. The
coupling itself stays on-device end to end (void-fraction deposit, drag, reaction, and the DEM force
buffer are all zero-copy device views); the only per-frame host traffic is the ~12 MB of positions the
renderer needs.

## Adapt this yourself

- **Change the drag law.** `drag="wen_yu"`, `"di_felice"`, `"schiller_naumann"`, `"stokes"` — or
  `"gidaspow"` here. Each is a device-inline correlation in `coupling/src/drag.hpp`; the porous mode
  applies the Model-B `β/ε` conversion to any of them.
- **Change the vessel.** The wall SDFs are ordinary functions — swap the cylinder for a cone (spouted
  bed), add a draft tube, or make the distributor a perforated plate.
- **Go multi-rank.** Build the `peclet-*-mpi` modules, pass `flow.init_mpi` / `dem.init_mpi` the same
  decomposition, and `CfdDem` runs the distributed deposit-fold + gather + DEM step.
- **Taller, wider, denser.** The step cost is dominated by the DEM at ~0.5 s per million grains; the
  fluid grid is far from its limit.

## Reproduce this

```bash
# CPU (OpenMP) — flow + dem + the coupling module, built from source against a Kokkos prefix:
tools/bootstrap_deps.sh host-openmp                      # one-time Kokkos/ArborX prefix
for m in flow dem coupling; do
  cmake -S $m -B $m/build -DCMAKE_PREFIX_PATH="$PWD/extern/install/host-openmp"; cmake --build $m/build -j
done
PECLET_LOCAL_BUILD="$PWD/flow/build:$PWD/dem/build:$PWD/coupling/build" \
  quarto render examples/fluidized-bed/index.qmd
# GPU (needed for the million-grain section): point CMAKE_PREFIX_PATH at
# extern/install/nvidia-cuda instead (nvcc on PATH) and pip install cupy-cuda13x pyvista imageio-ffmpeg.
```